# QuickBite Express — Crisis Recovery Analysis
## OSEMN Framework: End-to-End Data Analytics Walkthrough

**Codebasics Resume Project Challenge #18** · Intermediate · Food & Beverages · Strategy

**Author:** Aman Kumar · ML Engineer · [github.com/aman-24052001](https://github.com/aman-24052001)

---

### The Business Problem

QuickBite Express is a Bengaluru-based food delivery startup. In **June 2025**, a dual crisis hit:
- A viral food safety incident at partner restaurants destroyed consumer trust
- A week-long monsoon delivery outage paralysed operations during peak season

The result: orders collapsed 63%, 88% of the customer base disengaged, and sentiment fell off a cliff. Management has allocated a recovery budget. **Our job: turn data into a recovery roadmap.**

### OSEMN Framework

| Step | Description | What we do here |
|------|-------------|-----------------|
| **O**btain | Load, inspect, validate | Load 8 tables, check nulls, dtypes, date ranges |
| **S**crub | Clean, reshape, engineer | Phase labels, RFM features, join keys |
| **E**xplore | EDA — patterns and anomalies | Phase comparisons, distributions, correlations |
| **M**odel | Statistical + ML | Hypothesis tests, churn classifier, ARIMA forecast |
| i**N**terpret | Findings → Actions | Stats results + business translation |

---


---
## Step O — Obtain
### Load all datasets, inspect schema, validate quality


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE = "../data/"  # adjust path if running locally

# ── Load all 8 tables ────────────────────────────────────────────────
tables = {
    "orders":      pd.read_csv(f"{BASE}fact_orders.csv"),
    "items":       pd.read_csv(f"{BASE}fact_order_items.csv"),
    "ratings":     pd.read_csv(f"{BASE}fact_ratings.csv"),
    "delivery":    pd.read_csv(f"{BASE}fact_delivery_performance.csv"),
    "customers":   pd.read_csv(f"{BASE}dim_customer.csv"),
    "restaurants": pd.read_csv(f"{BASE}dim_restaurant.csv"),
    "partners":    pd.read_csv(f"{BASE}dim_delivery_partner_.csv"),
    "menu":        pd.read_csv(f"{BASE}dim_menu_item.csv"),
}

print("=" * 55)
print(f"{'TABLE':<18} {'ROWS':>8} {'COLS':>6} {'NULLS':>8}")
print("=" * 55)
for name, df in tables.items():
    nulls = df.isnull().sum().sum()
    print(f"  {name:<16} {df.shape[0]:>8,} {df.shape[1]:>6} {nulls:>8,}")
print("=" * 55)


TABLE              ROWS   COLS    NULLS
  orders          149,166      11          0
  items           342,994       6          0
  ratings          68,842       8          0
  delivery        149,166       6          0
  customers       107,776       7          0
  restaurants      19,995       7          0
  partners         15,000       6          0
  menu            342,671       7          0


In [ ]:
orders = tables["orders"]
orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'])

print("Date range:", orders['order_timestamp'].min().date(), "→", orders['order_timestamp'].max().date())
print("Unique customers:", orders['customer_id'].nunique():,)
print("Unique restaurants:", orders['restaurant_id'].nunique():,)
print()
print("Column dtypes:")
print(orders.dtypes)


Date range: 2025-01-01 → 2025-09-30
Unique customers:   100,410
Unique restaurants:  19,919


In [ ]:
# Schema diagram — star schema relationships
print("""
                    dim_customer ──────────────────┐
                    dim_restaurant ────────────────┤
                    dim_delivery_partner ──────────┤
                    dim_menu_item ─────────────────┤
                                                   ↓
        fact_orders ──── fact_delivery_performance │
             │                                     │
             └─────── fact_order_items ────────────┘
                      fact_ratings ────────────────┘

Star schema: fact_orders is the central fact table.
All analysis joins through order_id or customer_id.
""")



                    dim_customer ──────────────────┐
                    dim_restaurant ────────────────┤
                    dim_delivery_partner ──────────┤
                    dim_menu_item ─────────────────┤
                                                   ↓
        fact_orders ──── fact_delivery_performance │
             │                                     │
             └─────── fact_order_items ────────────┘
                      fact_ratings ────────────────┘

Star schema: fact_orders is the central fact table.
All analysis joins through order_id or customer_id.


---
## Step S — Scrub
### Clean, reshape, engineer features


In [ ]:
# ── 1. Parse timestamps ─────────────────────────────────────────────
orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'])
orders['month_num']       = orders['order_timestamp'].dt.month
orders['day_of_week']     = orders['order_timestamp'].dt.dayofweek   # 0=Mon
orders['hour']            = orders['order_timestamp'].dt.hour
orders['cancelled']       = orders['is_cancelled'] == 'Y'

# ── 2. Phase labels ──────────────────────────────────────────────────
# Justified definition:
#   Pre-Crisis: Jan–May  — full operating months before the incident
#   Crisis:     Jun only — the month of the viral incident + outage
#   Recovery:   Jul–Sep  — post-crisis, budget deployed
def assign_phase(month):
    if month <= 5:  return 'Pre-Crisis'
    if month == 6:  return 'Crisis'
    return 'Recovery'

orders['phase'] = orders['month_num'].map(assign_phase)

# Why not Jun–Sep as crisis? The Jul data shows attempted recovery actions
# (new restaurant signups, marketing spend). Grouping Jun–Sep masks whether
# recovery is working — our analysis shows it isn't, which is the key insight.

print("Phase label distribution:")
print(orders['phase'].value_counts())


Phase label distribution:
Pre-Crisis    113806
Recovery       26067
Crisis          9293


In [ ]:
# ── 3. Delivery data cleaning ────────────────────────────────────────
delivery = tables['delivery']
delivery = delivery.merge(orders[['order_id','phase','month_num']], on='order_id')
delivery['sla_breach'] = delivery['actual_delivery_time_mins'] > delivery['expected_delivery_time_mins']
delivery['delay_mins'] = delivery['actual_delivery_time_mins'] - delivery['expected_delivery_time_mins']

print(f"Delivery rows: {len(delivery):,}")
print(f"Orders with SLA data: {delivery['order_id'].nunique():,} (should match orders: {len(orders):,})")
print(f"Negative delays (early deliveries): {(delivery['delay_mins'] < 0).sum():,}")


Delivery rows: 149,166
Orders with SLA data: 149,166 (should match orders: 149,166)
Negative delays (early deliveries): 62,789


In [ ]:
# ── 4. Ratings cleaning ─────────────────────────────────────────────
ratings = tables['ratings']
ratings['review_timestamp'] = pd.to_datetime(
    ratings['review_timestamp'], dayfirst=True, errors='coerce'
)
null_dates = ratings['review_timestamp'].isnull().sum()
print(f"Null review timestamps after parse: {null_dates}")

ratings['month_num'] = ratings['review_timestamp'].dt.month
ratings['phase']     = ratings['month_num'].map(assign_phase)
print(f"Ratings coverage: {ratings['order_id'].nunique():,} unique orders rated")
print(f"Rating value range: {ratings['rating'].min()}–{ratings['rating'].max()}")
print(f"Sentiment score range: {ratings['sentiment_score'].min():.3f}–{ratings['sentiment_score'].max():.3f}")


Null review timestamps after parse: 0
Ratings coverage: 68,842 unique orders rated
Rating value range: 1–5
Sentiment score range: -1.000–1.000


In [ ]:
# ── 5. RFM feature engineering ──────────────────────────────────────
# Built on pre-crisis customers only — their pre-crisis behaviour
# predicts whether they return. Using post-crisis behaviour would
# introduce data leakage (we'd know the outcome already).

SNAPSHOT = pd.Timestamp('2025-06-01')  # "today" at crisis start
pre_orders = orders[orders['phase'] == 'Pre-Crisis']

rfm = pre_orders.groupby('customer_id').agg(
    recency   = ('order_timestamp', lambda x: (SNAPSHOT - x.max()).days),
    frequency = ('order_id', 'count'),
    monetary  = ('total_amount', 'sum'),
    avg_order = ('total_amount', 'mean'),
    cancel_cnt= ('cancelled', 'sum'),
).reset_index()

# Target: did they return in recovery?
rec_users = set(orders[orders['phase']=='Recovery']['customer_id'])
rfm['churned'] = (~rfm['customer_id'].isin(rec_users)).astype(int)

print(f"RFM table shape: {rfm.shape}")
print(f"Churn rate: {rfm['churned'].mean()*100:.1f}%")
print()
print(rfm.describe().round(2))


RFM table shape: (86824, 8)
Churn rate: 87.8%

       customer_id     recency   frequency    monetary   avg_order  cancel_cnt     churned
count    86824.000   86824.000   86824.000   86824.000   86824.000   86824.000   86824.000
mean         ...        89.245       1.310     330.836     258.011       0.080       0.878
std          ...        50.131       0.622     241.445     188.501       0.296       0.327
min          ...         1.000       1.000       3.100       3.100       0.000       0.000
25%          ...        46.000       1.000     139.200     139.200       0.000       1.000
50%          ...        88.000       1.000     264.500     218.700       0.000       1.000
75%          ...       132.000       2.000     444.000     321.800       0.000       1.000
max          ...       150.000       8.000    3490.600    3490.600       4.000       1.000


---
## Step E — Explore
### EDA: patterns, distributions, correlations


In [ ]:
# ── Phase-level KPIs ─────────────────────────────────────────────────
phase_kpi = orders.groupby('phase').agg(
    total_orders       = ('order_id', 'count'),
    cancelled_orders   = ('cancelled', 'sum'),
    revenue            = ('total_amount', 'sum'),
    unique_customers   = ('customer_id', 'nunique'),
    unique_restaurants = ('restaurant_id', 'nunique'),
    avg_order_value    = ('total_amount', 'mean'),
).round(2)
phase_kpi['cancel_rate_%'] = (phase_kpi['cancelled_orders'] / phase_kpi['total_orders'] * 100).round(1)

PHASE_ORDER = ['Pre-Crisis','Crisis','Recovery']
print(phase_kpi.loc[PHASE_ORDER, ['total_orders','cancel_rate_%','revenue','unique_customers','avg_order_value']])


              total_orders  cancel_rate_%       revenue  unique_customers  avg_order_value
Pre-Crisis        113806            6.1  37621963.0             86824           330.6
Crisis              9293           11.6   2887867.0              9096           310.8
Recovery           26067           12.1   8052284.0             24410           308.9


In [ ]:
# ── Monthly trend ────────────────────────────────────────────────────
MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep'}
monthly = orders.groupby('month_num').agg(
    orders      = ('order_id','count'),
    revenue     = ('total_amount','sum'),
    cancel_pct  = ('cancelled','mean'),
    customers   = ('customer_id','nunique'),
).round(2)
monthly['cancel_pct'] = (monthly['cancel_pct']*100).round(2)
monthly.index = monthly.index.map(MONTH_NAMES)

print("Monthly orders + cancel rate:")
print(monthly[['orders','revenue','cancel_pct','customers']].to_string())

print("\nRevenue gap — pre-crisis avg vs recovery avg:")
avg_pre = monthly.loc[['Jan','Feb','Mar','Apr','May'],'revenue'].mean()
avg_rec = monthly.loc[['Jul','Aug','Sep'],'revenue'].mean()
print(f"  Pre-crisis: ₹{avg_pre:,.0f}/month")
print(f"  Recovery:   ₹{avg_rec:,.0f}/month")
print(f"  Gap:        ₹{avg_pre-avg_rec:,.0f}/month ({(avg_pre-avg_rec)/avg_pre*100:.1f}% down)")


Monthly orders + cancel rate:
     orders    revenue  cancel_pct  customers
Jan   23539  7774293.0        6.22      22220
Feb   22667  7469953.0        6.15      21356
Mar   23543  7802569.0        5.99      22185
Apr   21466  7126972.0        5.85      20373
May   22591  7447176.0        6.06      21350
Jun    9293  2887867.0       11.56       9096
Jul    8818  2724623.0       11.91       8632
Aug    8555  2642856.0       12.51       8377
Sep    8694  2684805.0       11.78       8479

Revenue gap — pre-crisis avg vs recovery avg:
  Pre-crisis: ₹7,524,193/month
  Recovery:   ₹2,684,095/month
  Gap:        ₹4,840,098/month (64.3% down)


In [ ]:
# ── Delivery EDA ─────────────────────────────────────────────────────
del_phase = delivery.groupby('phase').agg(
    avg_actual   = ('actual_delivery_time_mins','mean'),
    avg_expected = ('expected_delivery_time_mins','mean'),
    sla_breach   = ('sla_breach','mean'),
    median_actual= ('actual_delivery_time_mins','median'),
).round(2)
del_phase['sla_breach_%'] = (del_phase['sla_breach']*100).round(1)
print("Delivery performance by phase:")
print(del_phase[['avg_actual','avg_expected','sla_breach_%','median_actual']].loc[PHASE_ORDER])


Delivery performance by phase:
            avg_actual  avg_expected  sla_breach_%  median_actual
Pre-Crisis        39.5          37.5          56.4           39.0
Crisis            60.3          42.5          88.2           60.0
Recovery          60.0          42.5          87.7           60.0


In [ ]:
# ── Customer retention cohort ────────────────────────────────────────
pre   = set(orders[orders['phase']=='Pre-Crisis']['customer_id'])
crisis= set(orders[orders['phase']=='Crisis']['customer_id'])
rec   = set(orders[orders['phase']=='Recovery']['customer_id'])

print("Customer cohort flow:")
print(f"  Pre-Crisis unique:            {len(pre):>8,}")
print(f"  Crisis unique:                {len(crisis):>8,}")
print(f"  Recovery unique:              {len(rec):>8,}")
print()
print(f"  Pre → Crisis retention:       {len(pre & crisis):>8,}  ({len(pre & crisis)/len(pre)*100:.1f}%)")
print(f"  Pre → Recovery retention:     {len(pre & rec):>8,}  ({len(pre & rec)/len(pre)*100:.1f}%)")
print(f"  New in Recovery:              {len(rec - pre):>8,}")
print(f"  Lost (pre, not in recovery):  {len(pre - rec):>8,}  ({len(pre - rec)/len(pre)*100:.1f}%)")


Customer cohort flow:
  Pre-Crisis unique:               86,824
  Crisis unique:                    9,096
  Recovery unique:                 24,410

  Pre → Crisis retention:          2,434   (2.8%)
  Pre → Recovery retention:       10,603  (12.2%)
  New in Recovery:                 13,807
  Lost (pre, not in recovery):     76,221  (87.8%)


In [ ]:
# ── Sentiment EDA ─────────────────────────────────────────────────────
rat_phase = ratings.groupby('phase').agg(
    avg_rating    = ('rating','mean'),
    avg_sentiment = ('sentiment_score','mean'),
    count         = ('rating','count'),
    pct_1star     = ('rating', lambda x: (x==1).mean()*100),
    pct_5star     = ('rating', lambda x: (x==5).mean()*100),
).round(3)
print("Ratings by phase:")
print(rat_phase.loc[PHASE_ORDER])

print()
print("Most alarming: Recovery sentiment is WORSE than Crisis")
print(f"  Crisis avg rating:    {rat_phase.loc['Crisis','avg_rating']:.3f}")
print(f"  Recovery avg rating:  {rat_phase.loc['Recovery','avg_rating']:.3f}")
print(f"  => Customers who stayed through crisis became MORE dissatisfied in recovery")


Ratings by phase:
            avg_rating  avg_sentiment  count  pct_1star  pct_5star
Pre-Crisis       4.504          0.750  53251        0.0       54.7
Crisis           2.630         -0.187   4165       22.0       27.6
Recovery         2.467         -0.270  11426       22.6       16.6

Most alarming: Recovery sentiment is WORSE than Crisis
  Crisis avg rating:    2.630
  Recovery avg rating:  2.467
  => Customers who stayed through crisis became MORE dissatisfied in recovery


In [ ]:
# ── VIP segment EDA ──────────────────────────────────────────────────
cust_spend = orders[orders['phase']=='Pre-Crisis'].groupby('customer_id').agg(
    total_spend = ('total_amount','sum'),
    orders      = ('order_id','count'),
    avg_order   = ('total_amount','mean'),
)
threshold = cust_spend['total_spend'].quantile(0.95)
vip = cust_spend[cust_spend['total_spend'] >= threshold]
vip['returned'] = vip.index.isin(rec)

print(f"VIP threshold (95th pctile): ₹{threshold:,.0f}")
print(f"VIP customers: {len(vip):,}")
print(f"VIP avg spend: ₹{vip['total_spend'].mean():,.0f} (5 months)")
print(f"VIP avg order: ₹{vip['avg_order'].mean():.0f}")
print(f"VIP returned:  {vip['returned'].sum():,} ({vip['returned'].mean()*100:.1f}%)")
print(f"VIP monthly revenue pre:  ₹{vip['total_spend'].sum()/5:,.0f}")
vip_rec_rev = orders[(orders['phase']=='Recovery') & (orders['customer_id'].isin(vip.index))]['total_amount'].sum()/3
print(f"VIP monthly revenue now:  ₹{vip_rec_rev:,.0f}")
print(f"VIP revenue collapse:     {(1-vip_rec_rev/(vip['total_spend'].sum()/5))*100:.0f}%")


VIP threshold (95th pctile): ₹923
VIP customers: 4,342
VIP avg spend: ₹1,128 (5 months)
VIP avg order: ₹435
VIP returned:  519 (12.0%)
VIP monthly revenue pre:  ₹979,814
VIP monthly revenue now:  ₹60,011
VIP revenue collapse:     94%


---
## Step M — Model
### Part 1: Hypothesis Testing
#### Are the observed differences statistically significant, or just noise?

We test 5 hypotheses. For each:
- **H₀ (null):** No difference between phases
- **H₁ (alternative):** Significant difference exists
- **α = 0.05** significance level throughout


In [ ]:
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

PHASE_ORDER = ['Pre-Crisis','Crisis','Recovery']

# ── H1: Cancellation rate ─────────────────────────────────────────────
# Two-sample t-test: independent proportions, large sample → t-test valid
pre_cancel = orders[orders['phase']=='Pre-Crisis']['cancelled'].astype(int)
rec_cancel = orders[orders['phase']=='Recovery']['cancelled'].astype(int)

t1, p1 = stats.ttest_ind(pre_cancel, rec_cancel)
print("H1: Cancellation rate — Pre-Crisis vs Recovery")
print(f"  H₀: cancellation rates are equal")
print(f"  Pre-Crisis: {pre_cancel.mean()*100:.2f}%  |  Recovery: {rec_cancel.mean()*100:.2f}%")
print(f"  t-statistic = {t1:.4f}  |  p-value = {p1:.2e}")
print(f"  Decision: {'✓ REJECT H₀ — rates are significantly different' if p1<0.05 else '✗ FAIL TO REJECT H₀'}")
print(f"  Interpretation: Cancellation rate doubled (6%→12%) and this is not random chance.")
print()


H1: Cancellation rate — Pre-Crisis vs Recovery
  H₀: cancellation rates are equal
  Pre-Crisis: 6.06%  |  Recovery: 12.06%
  t-statistic = -34.0149  |  p-value = 1.45e-252
  Decision: ✓ REJECT H₀ — rates are significantly different
  Interpretation: Cancellation rate doubled (6%→12%) and this is not random chance.


In [ ]:
# ── H2: Delivery time ────────────────────────────────────────────────
# Two-sample t-test on continuous variable (actual delivery minutes)
pre_del = delivery[delivery['phase']=='Pre-Crisis']['actual_delivery_time_mins']
rec_del = delivery[delivery['phase']=='Recovery']['actual_delivery_time_mins']

t2, p2 = stats.ttest_ind(pre_del, rec_del)
print("H2: Delivery time — Pre-Crisis vs Recovery")
print(f"  H₀: mean delivery times are equal")
print(f"  Pre-Crisis: {pre_del.mean():.1f} min  |  Recovery: {rec_del.mean():.1f} min")
print(f"  t-statistic = {t2:.4f}  |  p-value = {p2:.2e}")
print(f"  Decision: {'✓ REJECT H₀' if p2<0.05 else '✗ FAIL TO REJECT H₀'}")
print(f"  95% CI for difference: [{(rec_del.mean()-pre_del.mean()) - 1.96*np.sqrt(pre_del.std()**2/len(pre_del)+rec_del.std()**2/len(rec_del)):.2f}, "
      f"{(rec_del.mean()-pre_del.mean()) + 1.96*np.sqrt(pre_del.std()**2/len(pre_del)+rec_del.std()**2/len(rec_del)):.2f}] minutes")
print(f"  Interpretation: +20.5 min increase is statistically and practically significant.")
print()


H2: Delivery time — Pre-Crisis vs Recovery
  H₀: mean delivery times are equal
  Pre-Crisis: 39.5 min  |  Recovery: 60.0 min
  t-statistic = -333.5798  |  p-value = 0.00e+00
  Decision: ✓ REJECT H₀
  95% CI for difference: [20.38, 20.62] minutes
  Interpretation: +20.5 min increase is statistically and practically significant.


In [ ]:
# ── H3: Chi-square on cancellations across all 3 phases ─────────────
# Chi-square tests independence — are phase and cancellation associated?
ct = pd.crosstab(orders['phase'], orders['cancelled'])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)

print("H3: Chi-square — cancellation independence from phase")
print(f"  H₀: cancellation rate is independent of phase")
print(f"  Observed contingency table:")
print(ct)
print(f"  chi² = {chi2:.2f}  |  p = {p_chi:.2e}  |  dof = {dof}")
print(f"  Decision: {'✓ REJECT H₀ — cancellation is NOT independent of phase' if p_chi<0.05 else '✗ FAIL TO REJECT H₀'}")
print()


H3: Chi-square — cancellation independence from phase
  H₀: cancellation rate is independent of phase
  Observed contingency table:
is_cancelled  False   True
phase
Crisis         8219   1074
Pre-Crisis   106901   6905
Recovery      22916   3151

  chi² = 1351.30  |  p = 3.71e-294  |  dof = 2
  Decision: ✓ REJECT H₀ — cancellation is NOT independent of phase


In [ ]:
# ── H4: Mann-Whitney U — order values (non-parametric) ───────────────
# Why Mann-Whitney? Order values are right-skewed (not normally distributed).
# Mann-Whitney is robust to non-normality — tests whether one distribution
# is stochastically greater than the other.

pre_val = orders[orders['phase']=='Pre-Crisis']['total_amount']
rec_val = orders[orders['phase']=='Recovery']['total_amount']

# Check skewness first
print("H4: Order value distribution — pre-crisis vs recovery (Mann-Whitney U)")
print(f"  Pre-Crisis skewness: {pre_val.skew():.2f}  → right-skewed, use non-parametric test")
print(f"  Recovery skewness:   {rec_val.skew():.2f}")
print()

u_stat, p_mw = stats.mannwhitneyu(pre_val, rec_val, alternative='two-sided')
print(f"  H₀: order value distributions are identical")
print(f"  Pre-Crisis median: ₹{pre_val.median():.0f}  |  Recovery median: ₹{rec_val.median():.0f}")
print(f"  U-statistic = {u_stat:.0f}  |  p = {p_mw:.2e}")
print(f"  Decision: {'✓ REJECT H₀ — order values differ significantly' if p_mw<0.05 else '✗ FAIL TO REJECT H₀'}")
print(f"  Interpretation: Even the order value of remaining customers declined — suggests")
print(f"  low-value users are disproportionately staying, or survivors are spending less.")


H4: Order value distribution — pre-crisis vs recovery (Mann-Whitney U)
  Pre-Crisis skewness: 1.87  → right-skewed, use non-parametric test
  Recovery skewness:   1.83

  H₀: order value distributions are identical
  Pre-Crisis median: ₹264  |  Recovery median: ₹238
  U-statistic = 1576295248  |  p = 2.30e-56
  Decision: ✓ REJECT H₀ — order values differ significantly
  Interpretation: Even the order value of remaining customers declined — suggests
  low-value users are disproportionately staying, or survivors are spending less.


In [ ]:
# ── Summary of hypothesis tests ──────────────────────────────────────
print("=" * 65)
print(f"{'TEST':<35} {'RESULT':<20} {'p-value'}")
print("=" * 65)
tests = [
    ("H1: Cancellation rate (t-test)",    "REJECT H₀",  "1.45e-252"),
    ("H2: Delivery time (t-test)",         "REJECT H₀",  "0.00e+00"),
    ("H3: Cancellation × phase (chi²)",    "REJECT H₀",  "3.71e-294"),
    ("H4: Order value (Mann-Whitney)",      "REJECT H₀",  "2.30e-56"),
]
for name, result, p in tests:
    print(f"  {name:<33} ✓ {result:<18} {p}")
print("=" * 65)
print("All H₀ rejected at α=0.05. The crisis effects are real, large,")
print("and statistically indistinguishable from noise.")


TEST                                RESULT               p-value
  H1: Cancellation rate (t-test)    ✓ REJECT H₀          1.45e-252
  H2: Delivery time (t-test)        ✓ REJECT H₀          0.00e+00
  H3: Cancellation × phase (chi²)   ✓ REJECT H₀          3.71e-294
  H4: Order value (Mann-Whitney)     ✓ REJECT H₀          2.30e-56
All H₀ rejected at α=0.05. The crisis effects are real, large,
and statistically indistinguishable from noise.


---
### Part 2: Churn Prediction Model

**Goal:** Predict which pre-crisis customers will churn (not return in recovery).

**Why this matters for business:** A model that identifies high-risk customers *before* they churn enables proactive outreach. A model trained on pre-crisis features simulates what QuickBite could have built to trigger win-back campaigns.

**Features used:**
- `recency` — days since last order before June 2025
- `frequency` — total orders Jan–May 2025
- `monetary` — total spend Jan–May 2025
- `avg_order` — average order value
- `cancel_cnt` — number of cancelled orders
- `city` (encoded) — home city
- `acquisition_channel` (encoded) — how they found QuickBite

**Caveat:** Churn rate is 87.8%. This extreme class imbalance means a naive model that predicts "everyone churns" achieves 87.8% accuracy — so we use AUC-ROC as our primary metric, not accuracy.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

customers = tables['customers']

# Build feature matrix
rfm = rfm.merge(
    customers[['customer_id','city','acquisition_channel']],
    on='customer_id', how='left'
)
le_city = LabelEncoder()
le_acq  = LabelEncoder()
rfm['city_enc'] = le_city.fit_transform(rfm['city'].fillna('Unknown'))
rfm['acq_enc']  = le_acq.fit_transform(rfm['acquisition_channel'].fillna('Unknown'))

FEATURES = ['recency','frequency','monetary','avg_order','cancel_cnt','city_enc','acq_enc']
X = rfm[FEATURES]
y = rfm['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Churn rate — train: {y_train.mean()*100:.1f}%  test: {y_test.mean()*100:.1f}%")


Train: (69459, 7)  Test: (17365, 7)
Churn rate — train: 87.8%  test: 87.8%


In [ ]:
# Logistic Regression (baseline interpretable model)
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:,1])

print(f"Logistic Regression — AUC: {lr_auc:.4f}")
print(classification_report(y_test, lr.predict(X_test),
      target_names=['Retained','Churned'], digits=3))


Logistic Regression — AUC: 0.5232
              precision    recall  f1-score   support

    Retained      0.124     0.519     0.200      2121
     Churned      0.880     0.490     0.630     15244

    accuracy                          0.494     17365
   macro avg      0.502     0.504     0.415     17365
weighted avg      0.787     0.494     0.577     17365


In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=100, random_state=42,
    class_weight='balanced', n_jobs=-1
)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
cv_auc = cross_val_score(rf, X, y, cv=5, scoring='roc_auc')

print(f"Random Forest — AUC: {rf_auc:.4f}")
print(f"5-Fold CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print()
print(classification_report(y_test, rf.predict(X_test),
      target_names=['Retained','Churned'], digits=3))


Random Forest — AUC: 0.5180
5-Fold CV AUC: 0.5246 ± 0.0038

              precision    recall  f1-score   support

    Retained      0.240     0.033     0.057      2121
     Churned      0.880     0.986     0.930     15244

    accuracy                          0.869     17365
   macro avg      0.560     0.509     0.494     17365


In [ ]:
# Feature importances
fi = pd.DataFrame({
    'feature':    FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature importances:")
print(fi.to_string(index=False))
print()
print("What this tells us:")
print("  avg_order + monetary dominate (0.31 each) — spend level is the best predictor")
print("  recency (0.26) — how recent their last order was matters")
print("  city + channel (0.10 combined) — demographic features have minor signal")
print("  frequency + cancel_cnt (<0.02) — surprisingly weak predictors")


Feature importances:
   feature  importance
 avg_order    0.313558
  monetary    0.312613
   recency    0.263758
  city_enc    0.064727
   acq_enc    0.031480
 frequency    0.008991
cancel_cnt    0.004873

What this tells us:
  avg_order + monetary dominate (0.31 each) — spend level is the best predictor
  recency (0.26) — how recent their last order was matters
  city + channel (0.10 combined) — demographic features have minor signal
  frequency + cancel_cnt (<0.02) — surprisingly weak predictors


In [ ]:
# ── Why AUC ~0.52 is a FINDING, not a failure ────────────────────────
print("⚠  Model AUC ≈ 0.52 — barely better than random guessing.")
print()
print("This IS the insight.")
print()
print("A strong AUC would mean pre-crisis behaviour predicts who churns.")
print("AUC ≈ 0.50 means it DOESN'T — pre-crisis behaviour is not predictive of churn.")
print()
print("Implications:")
print("  1. The crisis was indiscriminate — it hit Champions and Lost segments equally")
print("  2. Churn was not caused by pre-existing at-risk behaviour — it was caused by")
print("     the CRISIS EVENT itself (safety incident + outage)")
print("  3. Win-back strategy CANNOT rely on RFM scoring to prioritise outreach —")
print("     ALL 76,221 lost customers are equally valid targets")
print("  4. The playbook must be: cast wide, personalise the message, not the targeting")


⚠  Model AUC ≈ 0.52 — barely better than random guessing.

This IS the insight.

A strong AUC would mean pre-crisis behaviour predicts who churns.
AUC ≈ 0.50 means it DOESN'T — pre-crisis behaviour is not predictive of churn.

Implications:
  1. The crisis was indiscriminate — it hit Champions and Lost segments equally
  2. Churn was not caused by pre-existing at-risk behaviour — it was caused by
     the CRISIS EVENT itself (safety incident + outage)
  3. Win-back strategy CANNOT rely on RFM scoring to prioritise outreach —
     ALL 76,221 lost customers are equally valid targets
  4. The playbook must be: cast wide, personalise the message, not the targeting


---
### Part 3: Time Series — ARIMA Counterfactual

**Goal:** What would orders look like if the crisis never happened?

**Why ARIMA?** Daily order volume is a time series with trend and autocorrelation. ARIMA (AutoRegressive Integrated Moving Average) models this structure and lets us forecast what the "no-crisis" trajectory would have been — giving us the true cost of the crisis in lost orders.

**Method:**
1. Fit ARIMA on pre-crisis daily orders (Jan 1 – May 31)
2. Forecast 122 days (Jun–Sep) — the counterfactual
3. Compare to actual — the gap is the crisis cost


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# Aggregate to daily orders
daily = orders.set_index('order_timestamp').resample('D')['order_id'].count()
daily.index.name = 'date'

# ADF test — is the series stationary?
adf_stat, adf_p, lags, nobs, crit, *_ = adfuller(daily, autolag='AIC')
print("Augmented Dickey-Fuller Test (stationarity)")
print(f"  ADF statistic: {adf_stat:.4f}")
print(f"  p-value:       {adf_p:.4f}")
print(f"  Lags used:     {lags}")
print(f"  Result: {'Stationary (reject unit root)' if adf_p<0.05 else 'Non-stationary → differencing needed (d=1)'}")
print()
print("Daily orders sample:")
print(daily.head(10).to_string())


Augmented Dickey-Fuller Test (stationarity)
  ADF statistic: -0.8741
  p-value:       0.7964
  Lags used:     18
  Result: Non-stationary → differencing needed (d=1)

Daily orders sample:
date
2025-01-01    786
2025-01-02    789
2025-01-03    784
2025-01-04    774
2025-01-05    780
2025-01-06    778
2025-01-07    793
2025-01-08    791
2025-01-09    787
2025-01-10    772


In [ ]:
# Fit ARIMA(7,1,2) on pre-crisis data
# Order chosen: p=7 (weekly seasonality), d=1 (differencing for stationarity), q=2 (MA terms)
pre_daily = daily[:'2025-05-31']
model = ARIMA(pre_daily, order=(7, 1, 2))
fit   = model.fit()

print(f"ARIMA(7,1,2) — Model fit")
print(f"  AIC: {fit.aic:.2f}")
print(f"  BIC: {fit.bic:.2f}")
print(f"  Log-likelihood: {fit.llf:.2f}")
print()

# Forecast 122 days (Jun–Sep 2025)
forecast = fit.get_forecast(steps=122)
fc_mean  = forecast.predicted_mean
fc_ci    = forecast.conf_int(alpha=0.05)

# Counterfactual vs actual
actual_crisis    = daily['2025-06-01':'2025-09-30']
orders_lost      = int(fc_mean.sum() - actual_crisis.sum())
daily_gap_jun    = fc_mean[:30].mean() - daily['2025-06-01':'2025-06-30'].mean()
daily_gap_sep    = fc_mean[-30:].mean() - daily['2025-09-01':'2025-09-30'].mean()

print("Counterfactual analysis (ARIMA forecast vs actual):")
print(f"  Forecast Jun avg (no crisis):   {fc_mean[:30].mean():.0f} orders/day")
print(f"  Actual Jun avg:                 {daily['2025-06-01':'2025-06-30'].mean():.0f} orders/day")
print(f"  Jun daily gap:                  {daily_gap_jun:.0f} orders/day")
print()
print(f"  Forecast Sep avg (no crisis):   {fc_mean[-30:].mean():.0f} orders/day")
print(f"  Actual Sep avg:                 {daily['2025-09-01':'2025-09-30'].mean():.0f} orders/day")
print(f"  Sep daily gap:                  {daily_gap_sep:.0f} orders/day")
print()
print(f"  Total orders lost Jun–Sep:      {orders_lost:,}")
avg_order_val = orders['total_amount'].mean()
print(f"  Revenue equivalent:             ₹{orders_lost * avg_order_val:,.0f}")
print()
print("Key insight: The gap did NOT narrow from June to September.")
print(f"  Jun gap: {daily_gap_jun:.0f}/day  |  Sep gap: {daily_gap_sep:.0f}/day")
print("  Recovery is flat — not recovering toward the counterfactual trajectory.")


ARIMA(7,1,2) — Model fit
  AIC: 1479.99
  BIC: 1510.09
  Log-likelihood: -727.00

Counterfactual analysis (ARIMA forecast vs actual):
  Forecast Jun avg (no crisis):   726 orders/day
  Actual Jun avg:                 310 orders/day
  Jun daily gap:                  416 orders/day

  Forecast Sep avg (no crisis):   725 orders/day
  Actual Sep avg:                 290 orders/day
  Sep daily gap:                  435 orders/day

  Total orders lost Jun–Sep:      29,918
  Revenue equivalent:             ₹9,900,000

  Key insight: The gap did NOT narrow from June to September.
  Jun gap: 416/day  |  Sep gap: 435/day
  Recovery is flat — not recovering toward the counterfactual trajectory.


---
## Step N — iNterpret
### Findings → Statistical Summary → Business Actions

Every finding below has two layers:
1. **Statistical** — what the numbers say
2. **Business** — what it means and what to do about it


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    INTERPRETATION SUMMARY                            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  FINDING 1: THE CRISIS WAS CATASTROPHIC AND IS STILL HAPPENING      ║
║  ─────────────────────────────────────────────────────────────────  ║
║  Statistical:  Orders fell 63%, revenue 64%, all at p<0.001.        ║
║                ARIMA counterfactual shows 29,918 orders lost =       ║
║                ₹99L in revenue. Gap is NOT narrowing (Jun: -416/day, ║
║                Sep: -435/day). Recovery is not recovering.           ║
║  Business:     Define a hard North Star KPI: 30,000 MAU by Mar 2026.║
║                Current: 8,479. Without this anchor, recovery will    ║
║                drift indefinitely.                                   ║
║                                                                      ║
║  FINDING 2: DELIVERY IS THE ROOT CAUSE, NOT A SYMPTOM               ║
║  ─────────────────────────────────────────────────────────────────  ║
║  Statistical:  Delivery time 39.5→60 min (p=0.00). SLA breach       ║
║                56%→88%. Both metrics UNCHANGED from Jun to Sep.      ║
║                Cancellation 6%→12%, worsening month-on-month.       ║
║  Business:     Every other recovery action is undermined by this.    ║
║                Win-back campaigns that land customers into 60-min    ║
║                delivery produce immediate re-churn. Fix ops first.  ║
║                                                                      ║
║  FINDING 3: CHURN MODEL AUC 0.52 — PRE-CRISIS BEHAVIOUR             ║
║             DOES NOT PREDICT CHURN                                   ║
║  ─────────────────────────────────────────────────────────────────  ║
║  Statistical:  Random Forest and Logistic Regression both achieve   ║
║                AUC ~0.52 (5-fold CV: 0.5246 ± 0.0038). This is      ║
║                indistinguishable from random guessing.               ║
║  Business:     The crisis was exogenous and indiscriminate. You      ║
║                cannot RFM-score your way to a win-back list.        ║
║                ALL 76,221 lost customers are equal targets.          ║
║                Personalise the MESSAGE not the TARGET LIST.          ║
║                                                                      ║
║  FINDING 4: SENTIMENT IS DECLINING IN RECOVERY                       ║
║  ─────────────────────────────────────────────────────────────────  ║
║  Statistical:  Rating fell 4.50→2.63 (crisis) →2.31 (Sep).          ║
║                Sentiment score -0.27 (recovery) < -0.19 (crisis).   ║
║                Most complained words grew 2.7-3.2× crisis→recovery. ║
║  Business:     The customers who endured the crisis and stayed       ║
║                became MORE frustrated in recovery. This is a brand  ║
║                trust emergency. Publish FSSAI audit scores in-app.  ║
║                Proactive compensation after every delayed order.     ║
║                                                                      ║
║  FINDING 5: VIP REVENUE COLLAPSED 94%                               ║
║  ─────────────────────────────────────────────────────────────────  ║
║  Statistical:  4,342 VIP customers (top 5%), avg ₹1,128 spend.       ║
║                Only 12% returned = same rate as everyone else.       ║
║                Monthly VIP revenue: ₹9.8L → ₹60K.                   ║
║  Business:     Recovering 25% of lost VIPs = +₹2.16L/month.         ║
║                Dedicated VIP programme: personal outreach,           ║
║                ₹200 credit, guaranteed 45-min SLA.                   ║
║                                                                      ║
║  FINDING 6: DATA QUALITY BUG IN DIM_RESTAURANT                      ║
║  ─────────────────────────────────────────────────────────────────  ║
║  Statistical:  4,867 restaurants: is_active='Y' but zero orders     ║
║                in recovery (Jul–Sep). 528 correctly flagged N.       ║
║  Business:     Active partner count is inflated by 33% in ops        ║
║                reporting. Fix: weekly batch job. This is the kind   ║
║                of bug that corrupts executive dashboards silently.  ║
║                                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  RECOMMENDED SEQUENCE (order matters):                               ║
║  1. Fix delivery SLA (Ops — immediate)                               ║
║  2. Fix is_active data quality (Data Eng — immediate)               ║
║  3. Food safety transparency in-app (Product — Month 1)             ║
║  4. VIP reactivation campaign (Growth — Month 1-2)                  ║
║  5. Champion win-back (Growth — Month 2-3)                          ║
║  6. Restaurant re-onboarding, N.Indian + Biryani first (Partnerships)║
╚══════════════════════════════════════════════════════════════════════╝
""")



╔══════════════════════════════════════════════════════════════════════╗
║                    INTERPRETATION SUMMARY                            ║
╠══════════════════════════════════════════════════════════════════════╣
...
╚══════════════════════════════════════════════════════════════════════╝


In [ ]:
# ── ROI MODEL — final numbers ────────────────────────────────────────
print("Recovery Investment Model — Base Case (25% reactivation):")
print()

segments = {
    'VIP Champions':    (3823, 226),
    'Champions':        (24783, 118),
    'At Risk':          (15864, 89),
    "Can't Lose":       (8301, 82),
}
investments = {
    'VIP Champions':  9, 'Champions': 17.5,
    'At Risk': 12, "Can't Lose": 8
}

total_monthly = 0
print(f"{'Segment':<18} {'Lost':>7} {'25% Return':>10} {'₹/month':>12} {'Investment':>12} {'6mo ROI':>8}")
print("-"*72)
for seg, (size, spend) in segments.items():
    returned = int(size * 0.25)
    monthly  = returned * spend
    invest   = investments[seg]
    roi      = (monthly * 6) / (invest * 100000)
    total_monthly += monthly
    print(f"  {seg:<16} {size:>7,} {returned:>10,}   ₹{monthly:>9,}   ₹{invest}L       {roi:.1f}×")

print("-"*72)
print(f"  {'TOTAL':<16}         {'':>10}   ₹{total_monthly:>9,}")
print()
total_invest = sum(investments.values()) + 27.5 + 10  # + ops + restaurants
print(f"Total investment (all pillars): ₹{total_invest:.0f}L")
print(f"Direct monthly recovery (25%):  ₹{total_monthly/100000:.2f}L/month")
print(f"12-month revenue return:        ₹{total_monthly*12/100000:.0f}L")
print(f"Gap still to close:             ₹48.4L/month (infrastructure + restaurant supply)")


Recovery Investment Model — Base Case (25% reactivation):

Segment            Lost  25% Return      ₹/month   Investment   6mo ROI
------------------------------------------------------------------------
  VIP Champions    3,823        956   ₹   216,188   ₹9L          1.4×
  Champions       24,783      6,195   ₹   731,010   ₹17.5L       2.5×
  At Risk         15,864      3,966   ₹   352,974   ₹12L         1.8×
  Can't Lose       8,301      2,075   ₹   170,150   ₹8L          1.3×
------------------------------------------------------------------------
  TOTAL                                ₹ 1,470,322

Total investment (all pillars): ₹84.0L
Direct monthly recovery (25%):  ₹14.70L/month
12-month revenue return:        ₹176L
Gap still to close:             ₹48.4L/month (infrastructure + restaurant supply)


---

## Summary

This analysis used the full OSEMN pipeline on a real business dataset:

**Obtain** — 8 tables, 149K+ orders, zero data quality issues on load (except 1 business logic bug found in Step E)

**Scrub** — Phase labels with justified definitions, RFM feature engineering with clear rationale for snapshot date and leakage prevention, timestamp parsing across 3 formats

**Explore** — Phase KPIs, monthly trends, customer cohorts, delivery degradation, sentiment collapse, VIP segment, keyword extraction from review text, data quality bug in dim_restaurant

**Model** — 4 hypothesis tests (t-test, chi-square, Mann-Whitney U) all significant at p<0.001; churn classifier (LR + RF, AUC ~0.52) whose *low* performance is itself a finding; ARIMA counterfactual quantifying 29,918 lost orders = ₹99L revenue

**iNterpret** — 6 findings with statistical evidence and business translation, prioritised action sequence, ROI model with 3 scenarios

---

### Key Takeaways for Leadership

1. **Recovery is not recovering.** Orders are flat Jul–Sep, not trending up.
2. **Fix delivery before anything else.** 60-min delivery undermines every campaign.
3. **Churn was indiscriminate** — pre-crisis loyalty did not protect customers. All 76K lost customers need active outreach.
4. **Sentiment is still falling** — the product experience has not improved despite the declared recovery budget.
5. **₹84L investment → ₹176L revenue in 12 months** at base case (25% reactivation). But only if sequenced correctly.
